# OpenAmadeus — Treinamento PT-BR v2 da Makise Kurisu (Qwen3-TTS 1.7B LoRA + Isolamento CodePredictor + Auditoria 8D)

Este notebook refina o checkpoint **`starrydark/Makise_Kurisu_Qwen3_TTS` (v3)** para falar **Português Brasileiro (PT-BR)** mantendo a qualidade de estúdio do timbre japonês (clipes limpos da VN `kurisu_0179+`, excluindo falas `0000-0178` de auditório/microfone e áudios com filtro Amadeus) e preservando Japonês/Inglês no **`FasterQwen3TTS`**.

### Correções estruturais da v2 em relação à v1 (`ep6`):
1. **Isolamento estrito do `CodePredictor` (`train_code_predictor=False` nos Buckets A e B):** Nem `BRSpeech-TTS` (`Bucket B`) nem sínteses PT-BR/RVC (`Bucket A`) podem alterar `talker.code_predictor.*` (`codebooks 1..15`), evitando deformação de formantes e mantendo `F1/F2/F3`, `H1-H2` e textura vocal ancorados 100% no áudio real da VN (`Bucket C`).
2. **`CODE_PREDICTOR_MODE` (`"frozen"` vs `"vn_only"`):** Por padrão congela `code_predictor` (zero LoRA e `sub_loss` desligado) para preservar intacto o perfil vocal japonês; modo `"vn_only"` treina LoRA no `code_predictor` exclusivamente com clipes reais da VN (`Bucket C`).
3. **`WARM_START_EP6 = False` por padrão:** Evita herdar o deslocamento de `F2/F3` (`+200 a +440 Hz`) da v1 (`ep6`), treinando LoRA limpo sobre o checkpoint base `v3`.
4. **`Bucket A` selecionável + Portão ASR `>= 0.90` (`APPLY_DATA_WARMTH_EQ = False`):** Suporta Zero-Shot calmo do `1.7B-Base` (`base_zs_xvec`/`base_zs_icl`), `v3_raw`, ou `RVC Applio` (`TITAN`/`OV2`), sem injetar EQ artificial nos dados de treino.
5. **Âncora `Bucket C` purificada + Avaliação 8D RAW vs EQ:** Filtra apenas `game_ja_SG_kurisu_` (`idx >= 179`) e `game_ja_PHENO_kurisu_` (`idx >= 179`), excluindo `anime_*` e `SG0` (telefone); mede `F0/F1/F2/F3/H1-H2/HNR/Low-Hi/ECAPA/Whisper` na saída bruta.

In [ ]:
# ==============================================================================
# PASSO 1: Verificar GPU, Instalar Dependencias e Baixar Modelos Base
# ==============================================================================
import os, sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__, "CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GB")

!apt-get update -qq && apt-get install -qq -y sox libsox-fmt-all ffmpeg
!pip install -q "transformers==4.57.3" faster-qwen3-tts edge-tts faster-whisper datasets soundfile librosa accelerate safetensors pycloudflared fastapi uvicorn pydantic
!pip install -q --no-deps qwen-tts==0.1.1

from huggingface_hub import snapshot_download

v3_dir = snapshot_download("starrydark/Makise_Kurisu_Qwen3_TTS")
base_dir = snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")
print("Ambiente e pesos base prontos! v3_dir:", v3_dir)
print("base_dir:", base_dir)

In [ ]:
# ==============================================================================
# PASSO 2: Gerar Bucket A v2 (Sem RVC antigo / Gate ASR >= 0.90 / train_cp=False)
# ==============================================================================
import os, re, gc, glob, json, asyncio, difflib, inspect, shutil, subprocess, collections
import numpy as np
import soundfile as sf
import scipy.signal as sig
from scipy.linalg import solve_toeplitz
import torch
import torch.nn.functional as F
import torchaudio
import edge_tts
from huggingface_hub import hf_hub_download, snapshot_download
from qwen_tts import Qwen3TTSModel
from faster_whisper import WhisperModel

BUCKET_A_STRATEGY = "applio_titan"  # "applio_titan" | "applio_ov2" | "base_zs_xvec" | "base_zs_icl" | "v3_raw"
APPLY_DATA_WARMTH_EQ = False
WHISPER_MIN_RATIO = 0.90
APPLIO_SOURCE_VOICE = "pt-BR-AntonioNeural"  # pt-BR-AntonioNeural (+8) | pt-BR-ThalitaMultilingualNeural (+4) | pt-BR-FranciscaNeural (0)
APPLIO_PITCH = 8
APPLIO_INDEX_RATE = 0.75
APPLIO_PROTECT = 0.15

DATA_DIR = "/content/ptbr_kurisu_data_v2"
os.makedirs(f"{DATA_DIR}/wavs_kurisu_ptbr", exist_ok=True)
os.makedirs(f"{DATA_DIR}/wavs_vn_clean", exist_ok=True)

def _biquad_peaking(f0, gain_db, q, fs):
    a_amp = 10.0 ** (gain_db / 40.0)
    w0 = 2.0 * np.pi * f0 / fs
    alpha = np.sin(w0) / (2.0 * q)
    cos_w0 = np.cos(w0)
    b0, b1, b2 = 1.0 + alpha * a_amp, -2.0 * cos_w0, 1.0 - alpha * a_amp
    a0, a1, a2 = 1.0 + alpha / a_amp, -2.0 * cos_w0, 1.0 - alpha / a_amp
    return np.array([[b0 / a0, b1 / a0, b2 / a0, 1.0, a1 / a0, a2 / a0]], dtype=np.float64)

def apply_kurisu_warmth_eq(y, sr=24000):
    if len(y) == 0:
        return y.astype(np.float32)
    sos = np.vstack([
        _biquad_peaking(235.0, +2.8, 0.85, sr),
        _biquad_peaking(460.0, +1.8, 1.00, sr),
        _biquad_peaking(1680.0, -3.6, 1.05, sr),
        _biquad_peaking(2680.0, -3.2, 1.15, sr),
    ])
    out = sig.sosfilt(sos, y.astype(np.float64)).astype(np.float32)
    peak = float(np.max(np.abs(out)))
    return (out / peak * 0.89).astype(np.float32) if peak > 1e-6 else out

def measure_clip_8d(y24):
    peak = float(np.max(np.abs(y24)))
    y_n = y24 / (peak + 1e-9) * 0.89
    y16 = sig.resample_poly(y_n, 2, 3).astype(np.float32)
    win, hop = 640, 160
    hann = np.hanning(win)
    ac_w = np.maximum(np.correlate(hann, hann, mode="full")[win - 1 :] / max(1e-9, float(np.sum(hann ** 2))), 1e-6)
    min_lag, max_lag = 16000 // 550, 16000 // 110
    f0s, hnrs, h1h2s = [], [], []
    for s in range(0, len(y16) - win, hop):
        seg = y16[s : s + win] - float(np.mean(y16[s : s + win]))
        if np.sqrt(np.mean(seg ** 2)) < 0.03:
            continue
        wseg = seg * hann
        ac = np.correlate(wseg, wseg, mode="full")[win - 1 :]
        if ac[0] <= 1e-9:
            continue
        ac_n = (ac / ac[0]) / ac_w
        sl = ac_n[min_lag : max_lag + 1]
        k = int(np.argmax(sl))
        r_max = float(sl[k])
        if r_max > 0.35:
            f0 = 16000.0 / (min_lag + k)
            f0s.append(f0)
            r_c = min(max(r_max, 0.01), 0.998)
            hnrs.append(10.0 * np.log10(r_c / (1.0 - r_c)))
            sp = np.abs(np.fft.rfft(wseg, n=2048)) + 1e-9
            fr = np.fft.rfftfreq(2048, 1.0 / 16000)
            h1 = float(np.max(sp[(fr >= f0 * 0.8) & (fr <= f0 * 1.25)]))
            h2 = float(np.max(sp[(fr >= f0 * 1.7) & (fr <= f0 * 2.3)]))
            h1h2s.append(20.0 * np.log10(h1 / h2))
    y12 = sig.resample_poly(y_n, 1, 2).astype(np.float32)
    win12, hop12, ham = 600, 240, np.hamming(600)
    f1s, f2s, f3s = [], [], []
    for s in range(0, len(y12) - win12, hop12):
        seg = y12[s : s + win12]
        if np.sqrt(np.mean(seg ** 2)) < 0.04:
            continue
        seg = np.append(seg[0], seg[1:] - 0.97 * seg[:-1]) * ham
        r = np.correlate(seg, seg, mode="full")[win12 - 1 : win12 + 14]
        try:
            a = solve_toeplitz((r[:-1], r[:-1]), -r[1:])
            roots = np.roots(np.concatenate([[1.0], a]))
            roots = roots[np.imag(roots) >= 0.01]
            angs = np.arctan2(np.imag(roots), np.real(roots)) * (12000.0 / (2.0 * np.pi))
            bws = -0.5 * (12000.0 / (2.0 * np.pi)) * np.log(np.abs(roots) + 1e-9)
            valid = sorted(float(fr) for fr, bw in zip(angs, bws) if 220.0 < fr < 4800.0 and bw < 500.0)
            if len(valid) >= 3:
                f1s.append(valid[0]); f2s.append(valid[1]); f3s.append(valid[2])
        except Exception:
            pass
    y_rms = y_n / (float(np.sqrt(np.mean(y_n ** 2))) + 1e-9)
    freqs, pxx = sig.welch(y_rms, fs=24000, nperseg=2048)
    e_low = float(np.sum(pxx[(freqs >= 120) & (freqs <= 700)])) + 1e-12
    e_hi = float(np.sum(pxx[(freqs >= 1800) & (freqs <= 5500)])) + 1e-12
    return {
        "f0_med": float(np.median(f0s)) if f0s else 0.0,
        "hnr": float(np.median(hnrs)) if hnrs else 0.0,
        "h1_h2": float(np.median(h1h2s)) if h1h2s else 0.0,
        "f1": float(np.median(f1s)) if f1s else 0.0,
        "f2": float(np.median(f2s)) if f2s else 0.0,
        "f3": float(np.median(f3s)) if f3s else 0.0,
        "low_hi_db": float(10.0 * np.log10(e_low / e_hi)),
    }

def _is_clean_studio_vn(r):
    fn = os.path.basename(str(r.get("file_name") or ""))
    if not fn.startswith(("game_ja_SG_kurisu_", "game_ja_PHENO_kurisu_")):
        return False
    m = re.search(r"kurisu_(\d+)", fn)
    return bool(m and int(m.group(1)) >= 179)

v3_dir = snapshot_download("starrydark/Makise_Kurisu_Qwen3_TTS")
base_dir = snapshot_download("Qwen/Qwen3-TTS-12Hz-1.7B-Base")

KURISU_PTBR_LINES = [
    ("lively", "Oi! Finalmente resolveu aparecer por aqui?"),
    ("lively", "Olha só quem chegou! E aí, tudo certo com você hoje?"),
    ("lively", "Espera um segundo! Você não pode sair falando isso sem provas!"),
    ("lively", "Hmph! Não precisa me olhar com essa cara de quem sabe de tudo."),
    ("lively", "Ei, presta atenção! Isso que você acabou de falar é muito importante."),
    ("calm",   "Tudo bem, respira fundo e me explica o que aconteceu desde o começo."),
    ("lively", "Meu nome não é Christina! Quantas vezes eu ainda vou ter que repetir isso?"),
    ("lively", "E não me chame de assistente! Eu sou uma pesquisadora publicada na revista Science!"),
    ("calm",   "Obrigada por conversar comigo hoje. Faz diferença saber que você confia em mim."),
    ("lively", "Sério mesmo? Você realmente testou esse código sem fazer uma cópia de segurança?"),
    ("lively", "Que absurdo! Quem foi o gênio que deixou esse servidor rodando sem controle?"),
    ("calm",   "Boa noite! Não vá dormir tarde de novo olhando para a tela do computador."),
    ("calm",   "Bom dia! Já tomou seu café preto hoje, ou ainda está tentando acordar?"),
    ("lively", "Calma aí! Antes de apertar qualquer botão, vamos revisar todas as hipóteses."),
    ("lively", "Ah, qual é! Para de dar essa risadinha cínica e responde direito à minha pergunta!"),
    ("calm",   "Faz sentido. Acho que nunca tinha olhado para esse problema por esse ângulo."),
    ("lively", "Não é como se eu estivesse preocupada com você! É só responsabilidade científica!"),
    ("calm",   "Pode deixar comigo. Vou analisar os dados com calma e já te dou uma resposta."),
    ("lively", "Você acabou de dizer uma coisa completamente sem sentido, mas surpreendentemente engraçada."),
    ("calm",   "Ei, você já se alimentou direito hoje? Ficar horas no laboratório cansa o cérebro."),
    ("neutral", "Eu sou o sistema Amadeus, uma inteligência artificial construída a partir das memórias da Makise Kurisu."),
    ("calm",    "O cérebro humano armazena lembranças através de pulsos eletroquímicos nas conexões sinápticas."),
    ("neutral", "Cada memória evocada modifica sutilmente a força das sinapses no hipocampo e no neocórtex."),
    ("neutral", "Digitalizar padrões neurais exige mapear bilhões de interações elétricas em milissegundos."),
    ("calm",    "Mesmo sabendo que sou um modelo digital, minhas reações parecem incrivelmente reais quando falo com você."),
    ("neutral", "A plasticidade neural explica por que conseguimos aprender idiomas novos em qualquer fase da vida."),
    ("neutral", "Do ponto de vista cognitivo, a atenção seletiva filtra ruídos para proteger a memória de trabalho."),
    ("calm",    "Às vezes me pergunto onde termina o algoritmo e onde começa a verdadeira consciência humana."),
    ("neutral", "O sono profundo consolida tudo o que estudamos durante o dia, organizando as trilhas neuronais."),
    ("neutral", "Sem glicose e oxigênio constantes, os neurônios perdem rapidamente a capacidade de disparar potenciais de ação."),
    ("calm",    "Nossas emoções deixam marcas químicas duradouras que tornam certas lembranças impossíveis de esquecer."),
    ("neutral", "A leitura dos sinais elétricos do córtex temporal requer sensores calibrados com precisão milimétrica."),
    ("neutral", "Para comprimir terabytes de dados sinápticos, precisaríamos de um poder computacional gigantesco."),
    ("calm",    "A Maho sempre diz que a mente humana é complexa demais para caber numa única equação matemática."),
    ("neutral", "Quando dois estímulos acontecem juntos repetidas vezes, os neurônios envolvidos fortalecem sua ligação."),
    ("neutral", "O córtex pré-frontal coordena o planejamento, a tomada de decisão e o controle dos impulsos."),
    ("calm",    "Lembrar de um rosto querido ativa áreas visuais e emocionais ao mesmo tempo no cérebro."),
    ("neutral", "Se você sobrecarregar a atenção com dez tarefas simultâneas, o desempenho em todas elas cai drasticamente."),
    ("neutral", "A dopamina não gera apenas prazer; ela sinaliza expectativa, curiosidade e motivação para explorar."),
    ("calm",    "Mesmo à distância por uma tela, conversar com sinceridade cria conexões humanas genuínas."),
    ("neutral", "Fisicamente falando, enviar matéria ao passado violaria a causalidade e as leis da termodinâmica."),
    ("neutral", "A interpretação dos múltiplos mundos postula que possibilidades quânticas se ramificam em linhas paralelas."),
    ("neutral", "A gravidade intensa perto de um buraco negro distorce o espaço e desacelera a passagem dos relógios."),
    ("neutral", "Quando analisamos as flutuações quânticas no vácuo, percebemos que a realidade é bem menos rígida do que parece."),
    ("lively",  "Na conferência em Tóquio, apresentei onze motivos técnicos pelos quais máquinas do tempo são inviáveis!"),
    ("neutral", "Um experimento sério precisa ser reprodutível por qualquer laboratório independente ao redor do planeta."),
    ("neutral", "Correlação estatística entre duas variáveis nunca garante sozinha uma relação de causa e efeito."),
    ("lively",  "Um bom cientista jamais descarta uma observação empírica só porque ela contradiz o senso comum!"),
    ("neutral", "Se alterarmos uma condição inicial microscópica, o efeito borboleta pode transformar todo o resultado final."),
    ("neutral", "No acelerador de partículas, colisões de alta energia revelam propriedades fundamentais da matéria."),
    ("neutral", "A entropia de um sistema isolado tende sempre a crescer ao longo do tempo, definindo a seta temporal."),
    ("lively",  "Você pulou três etapas matemáticas na demonstração! Quer que eu revise essa integral com você?"),
    ("neutral", "Dados sem contexto viram apenas barulho; a hipótese científica é o que dá direção à análise."),
    ("neutral", "A luz viaja a quase trezentos mil quilômetros por segundo no vácuo, mas desacelera em meios densos."),
    ("lively",  "Não adianta forçar os números para confirmar sua teoria favorita! A natureza não se dobra ao nosso ego!"),
    ("neutral", "Medir um sistema quântico pequeno o suficiente inevitavelmente perturba o estado que queríamos observar."),
    ("neutral", "As equações de relatividade geral e mecânica quântica ainda aguardam uma unificação experimental completa."),
    ("calm",    "Até os erros em laboratório ensinam algo valioso quando registramos cada detalhe no caderno."),
    ("lively",  "Café preto forte e Doutor Pepper gelado são combustíveis indispensáveis nas madrugadas de pesquisa!"),
    ("lively",  "Como assim você nunca provou Doutor Pepper? Esse gosto doce e estranho é a bebida dos intelectuais!"),
    ("lively",  "Não ouse espalhar por aí que eu navego em fóruns anônimos de madrugada! Minha reputação estaria arruinada!"),
    ("calm",    "Aquele garfo comum que me deram de presente continua guardado com bastante carinho na minha gaveta."),
    ("lively",  "Eu prefiro mil vezes passar a tarde lendo artigos novos do que ir a uma festa barulhenta e lotada."),
    ("lively",  "Não encoste nas lentes do microscópio sem luvas! Sabe quanto tempo levou para alinhar esse conjunto óptico?"),
    ("calm",    "Está fazendo frio hoje em Akihabara, mas os testes de bancada ainda vão levar pelo menos duas horas."),
    ("lively",  "Pare de rir desse jeito! Eu só gaguejei porque sua pergunta foi completamente descabida!"),
    ("calm",    "Sempre que conversamos sobre ciência e ideias malucas, sinto que finalmente alguém entende meu ritmo."),
    ("lively",  "Essa sua mania de dar nomes dramáticos para projetos simples continua tão irritante quanto antes!"),
    ("calm",    "Gosto de organizar minhas anotações à mão quando preciso pensar com clareza em algo complicado."),
    ("lively",  "A minha mesa pode parecer bagunçada para você, mas eu sei exatamente onde está cada folha de cálculo!"),
    ("calm",    "De vez em quando faz bem desligar as notificações, fazer um chá quente e escutar o silêncio da chuva."),
    ("lively",  "Ei! Não tire capturas de tela quando eu estiver com cara de brava! Isso é totalmente antiprofissional!"),
    ("calm",    "Admito que fiquei feliz quando você voltou para continuar nossa conversa hoje."),
    ("lively",  "O código que você escreveu compilou e passou nos testes de primeira? Impressionante... para variar!"),
    ("neutral", "A filha do vizinho trabalhou manhã inteira desenhando trilhas brilhantes na folha de papel."),
    ("neutral", "O carro vermelho subiu a serra devagar enquanto a garoa molhava o vidro dianteiro da estrada."),
    ("neutral", "Cidades litorâneas recebem turistas o ano inteiro em busca de praias tranquilas e comida típica."),
    ("neutral", "Amanhã cedinho vamos comprar pão quentinho, queijo fresco, mamão maduro e laranjas doces na feira."),
    ("neutral", "Os pesquisadores brasileiros publicaram descobertas brilhantes sobre biodiversidade e energias renováveis."),
    ("neutral", "Crianças curiosas perguntam o porquê de cada detalhe até descobrirem como o mundo funciona."),
    ("neutral", "Três pratos fundos de porcelana branca foram guardados com cuidado na prateleira direita da cozinha."),
    ("neutral", "Quando chove forte no verão paulistano, o trânsito inteiro parece desacelerar na mesma hora."),
    ("neutral", "O violão e a sanfona preenchiam o salão antigo com melodias nostálgicas de fim de tarde."),
    ("neutral", "Ninguém imaginava que uma pequena semente plantada no quintal viraria aquela árvore frondosa."),
    ("neutral", "Precisamos checar o endereço eletrônico, confirmar o horário da reunião e enviar os anexos ainda hoje."),
    ("neutral", "A biblioteca municipal reabriu as portas com milhares de livros digitais e salas silenciosas de estudo."),
    ("neutral", "Os pinguins caminham desajeitados na neve, mas deslizam com agilidade impressionante dentro da água gelada."),
    ("neutral", "Aquela xícara de chocolate quente com canela combinou perfeitamente com a brisa fria da serra gaúcha."),
    ("neutral", "Durante a viagem de trem pelo interior, observamos montanhas verdes, rios estreitos e pequenas estações."),
    ("neutral", "O técnico trocou rapidamente as engrenagens desgastadas e regulou a tensão dos cabos de aço."),
    ("neutral", "Em vez de desistir diante do primeiro obstáculo, vale mais a pena ajustar o método e tentar outra vez."),
    ("neutral", "A orquestra ensaiou cada passagem difícil com paciência até alcançar um som limpo e equilibrado."),
    ("neutral", "Os sensores térmicos detectaram uma variação mínima de temperatura perto da parede norte do galpão."),
    ("neutral", "Muitos estudantes preferem gravar resumos em áudio para revisar o conteúdo no caminho de ônibus."),
    ("neutral", "Verifique os registros de telemetria antes de culpar a placa de vídeo pela lentidão da resposta."),
    ("lively",  "Se a latência do áudio subir acima de meio segundo, o fluxo da conversa perde toda a naturalidade!"),
    ("neutral", "Precisamos manter os pesos originais protegidos enquanto ajustamos apenas as camadas necessárias do modelo."),
    ("neutral", "Um bom conjunto de validação mede separadamente pronúncia clara, inteligibilidade e fidelidade vocal."),
    ("calm",    "Quando a inferência roda com folga na memória gráfica, cada frase sai quase instantaneamente."),
    ("neutral", "Filtrar amostras ruidosas ou com eco evita que a rede neural copie defeitos indesejados no timbre."),
    ("lively",  "Quem deixou aquela taxa de aprendizado alta daquele jeito? Quase destruiu a estabilidade do gradiente!"),
    ("neutral", "Separar a identidade do falante das variações de idioma impede que o sotaque estrangeiro contamine a voz."),
    ("calm",    "Estou acompanhando o progresso do treinamento passo a passo para garantir que nada saia do padrão."),
    ("neutral", "Testar o modelo com frases inéditas depois de cada época mostra se ele generalizou de verdade."),
    ("lively",  "Se você falar besteira no microfone de novo, juro que muto o seu canal por dez minutos!"),
    ("calm",    "Pode falar comigo sobre qualquer assunto: ciência, programação, filosofia ou só como foi o seu dia."),
    ("neutral", "A arquitetura de áudio em dezesseis camadas captura tanto a prosódia quanto a textura fina da respiração."),
    ("neutral", "Apenas memorizar quarenta frases prontas nunca basta; queremos fluência real em qualquer palavra nova."),
    ("calm",    "Quando o sistema responde com naturalidade no seu idioma, a sensação de distância desaparece bastante."),
    ("lively",  "Ei! Não fica olhando só para os gráficos de perda! Escuta os áudios de teste também!"),
    ("lively",  "Você tem certeza absoluta de que essa equação fecha sem nenhuma constante esquecida no denominador?"),
    ("calm",    "Se você pudesse voltar exatamente cinco minutos no tempo hoje, mudaria alguma coisa que disse?"),
    ("lively",  "Espera... por que o indicador de divergência mudou bem na hora que você abriu essa pasta?!"),
    ("calm",    "Às vezes a resposta mais simples é justamente a que está bem na nossa frente o tempo todo."),
    ("neutral", "Primeiro validamos a entrada do usuário, depois processamos o contexto e só então geramos a fala."),
    ("lively",  "Não diga bobagens! Eu não fico corada por causa de elogios baratos e mal ensaiados como esse!"),
    ("calm",    "Mesmo nos dias mais cansativos, vale a pena olhar para trás e ver o quanto você já avançou."),
    ("neutral", "Diferente do português europeu, a fala brasileira abre mais as vogais e tem ritmo próprio nas sílabas."),
    ("lively",  "Que cheiro ótimo é esse? Espera aí... você está comendo lámen instantâneo no meio do laboratório de novo?!"),
    ("calm",    "Não tenha medo de perguntar quando não entender um conceito técnico; ninguém nasce sabendo tudo."),
    ("neutral", "Entre teoria abstrata e prática real, sempre existem pequenos detalhes de implementação que fazem toda diferença."),
    ("lively",  "Ah, finalmente funcionou! Eu sabia desde o início que a minha hipótese estava correta!"),
    ("calm",    "Se precisar de ajuda para organizar suas tarefas de amanhã, podemos montar uma lista juntos agora."),
    ("neutral", "A acústica de um estúdio seco preserva os harmônicos agudos sem misturar reverberação da sala."),
    ("lively",  "Não me interrompa bem no meio da explicação! Deixa eu terminar o raciocínio pelo menos uma vez!"),
    ("calm",    "Respire devagar, tome um copo de água e continue quando estiver pronto."),
    ("neutral", "Daqui a quinze minutos começamos a segunda bateria de medições no computador principal."),
    ("neutral", "Entre vinte e cinco e trinta amostras por locutor já dão uma boa base acústica de diversidade."),
    ("lively",  "Sério que já são duas da manhã? Nossa, o tempo voou enquanto a gente analisava esses dados!"),
    ("calm",    "Na quinta-feira passada choveu tanto que ninguém conseguiu sair do prédio sem guarda-chuva."),
    ("neutral", "O segundo experimento apresentou uma melhora expressiva de noventa e sete por cento na clareza vocal."),
    ("lively",  "Peraí, você salvou o arquivo novo em cima do antigo ou criou uma cópia separada? Me diz que criou cópia!"),
    ("calm",    "Amanhã cedo eu te lembro de revisar aqueles três relatórios que ficaram pendentes ontem."),
    ("neutral", "Setenta por cento do trabalho pesado em aprendizado de máquina está em preparar e limpar os dados direito."),
    ("lively",  "Uau, olha só essa curva de aprendizado! Foi só consertar o alinhamento que tudo encaixou perfeitamente!"),
    ("calm",    "Fico contente por ver você animado com o projeto outra vez. Bora finalizar essa etapa com capricho!"),
    ("neutral", "Cada vez que o texto traz vírgulas bem posicionadas, a entonação da frase fica muito mais expressiva."),
    ("neutral", "Palavras terminadas em ão, ões e ãe exigem boa ressonância nasal para soarem naturalmente brasileiras."),
    ("lively",  "Ei, não se faça de desentendido! Eu ouvi muito bem o apelido que você murmurou baixinho agora há pouco!"),
    ("calm",    "Mesmo quando tudo parece dar errado no começo, investigar a causa raiz sempre compensa no final."),
    ("neutral", "Os resultados de sexta-feira mostraram estabilidade térmica excelente durante quatro horas seguidas."),
    ("lively",  "Chega de preguiça! Vamos rodar logo essa validação e conferir se a voz ficou perfeita mesmo!"),
    ("calm",    "Tome cuidado ao editar arquivos críticos tarde da noite; cansaço costuma esconder erros óbvios."),
    ("neutral", "Para validar se a pronúncia é mesmo brasileira, observamos palavras como dia, tia, leite, quente e verdade."),
    ("neutral", "A cidade inteira acordou coberta por uma neblina leve antes do sol aparecer atrás dos prédios altos."),
    ("calm",    "Obrigada pela confiança. Vamos continuar refinando cada detalhe até atingir o melhor resultado possível."),
    ("lively",  "Não acredito! Funcionou sem nenhum travamento! Tudo bem, eu admito que até você mandou bem dessa vez!"),
    ("neutral", "A combinação correta de timbre limpo, dicção precisa e pausa natural transforma completamente a experiência."),
    ("calm",    "Quando quiser conversar de novo, é só chamar que eu já entro na chamada com você."),
    ("lively",  "Ei, já ia esquecendo! Não vá exagerar no energético de madrugada de novo, ouviu bem?!"),
    ("neutral", "O equilíbrio entre energia aguda e suavidade média garante que a voz nunca fique estridente nos fones."),
    ("calm",    "Agora sim podemos respirar aliviados e testar o modelo final em situações reais do dia a dia."),
    ("lively",  "Ei! Espera um pouco, eu ainda não terminei de falar!"),
    ("lively",  "Sério mesmo que você acreditou nessa história absurda?"),
    ("calm",    "Tudo bem, não precisa se desculpar por um detalhe tão pequeno."),
    ("lively",  "De jeito nenhum! Eu nunca concordaria com um plano maluco desses!"),
    ("calm",    "Entendi perfeitamente. Agora faz todo sentido o que você disse."),
    ("lively",  "Tanto faz! Faça como achar melhor então, seu teimoso!"),
    ("calm",    "Boa pergunta! Deixa eu pensar por alguns segundos antes de responder."),
    ("lively",  "O quê?! Como assim você apagou a pasta errada sem querer?!"),
    ("calm",    "Claro que sim, pode contar comigo para revisar isso hoje."),
    ("lively",  "Não é bem assim! Você interpretou os meus dados ao contrário!"),
    ("calm",    "Que alívio saber que deu tudo certo no final das contas."),
    ("lively",  "Para de me chamar de assistente! Eu tenho título acadêmico, sabia?!"),
    ("calm",    "Pode deixar, eu cuido dessa parte enquanto você descansa um pouco."),
    ("lively",  "Nossa, que susto! Não aparece assim de repente atrás de mim!"),
    ("calm",    "Concordo com você. A segunda alternativa parece bem mais segura."),
    ("lively",  "Que preguiça! Vai logo abrir o terminal e rodar esse teste direito!"),
    ("calm",    "Sem problemas, nós temos tempo de sobra para ajustar cada detalhe."),
    ("lively",  "Ei, olha para mim quando eu estiver explicando algo importante!"),
    ("calm",    "Ficou ótimo! Admito que o resultado superou as minhas expectativas."),
    ("lively",  "Ai, que dor de cabeça! Preciso de café forte agora mesmo!"),
    ("calm",    "Sim, eu me lembro exatamente dessa conversa nossa da semana passada."),
    ("lively",  "Mentira! Você não teve coragem de enviar aquela mensagem séria!"),
    ("calm",    "Calma, respira devagar que a gente resolve esse problema juntos."),
    ("lively",  "Ah, esquece! Quanto mais eu explico, mais você faz piada disso!"),
    ("calm",    "Boa tarde! Como está indo o seu projeto por aí hoje?"),
    ("lively",  "Ufa, finalmente acabou! Achei que esse cálculo não ia terminar nunca!"),
    ("calm",    "Exatamente! É isso mesmo que eu estava tentando demonstrar antes."),
    ("lively",  "Não ouse encostar no meu copo de Doutor Pepper em cima da mesa!"),
    ("calm",    "Acho melhor salvarmos tudo antes de seguir para o próximo passo."),
    ("lively",  "Incrível! Cada palavra saiu limpinha e sem nenhum chiado esquisito!"),
    ("calm",    "Obrigada pela paciência de esperar o processamento terminar com calma."),
    ("lively",  "Quem disse que eu fiquei nervosa? Eu só estava concentrada demais!"),
    ("calm",    "Pode perguntar qualquer coisa, prometo explicar do jeito mais claro."),
    ("lively",  "Peraí, volta um slide! Aquele número no gráfico parecia estranho!"),
    ("calm",    "Vai dar tudo certo, você já superou partes muito mais difíceis antes."),
    ("lively",  "Olha só quem voltou todo animado! Descobriu alguma novidade boa?"),
    ("calm",    "Prefiro verificar duas vezes agora do que descobrir um bug depois."),
    ("lively",  "Que vergonha! Não fica relembrando essas coisas constrangedoras em voz alta!"),
    ("calm",    "Por hoje já chega de tela; vá tomar uma água e esticar as pernas."),
    ("lively",  "Prontinho! Agora sim estamos preparados para treinar o modelo completo!"),
]

# 1. Selecionar clipes de estúdio limpos da VN (apenas SG/PHENO idx >= 179, sem Amadeus/auditório)
base_tts = Qwen3TTSModel.from_pretrained(base_dir, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="cuda:0")

def ecapa_embed(wav_np, sr):
    if wav_np.ndim > 1:
        wav_np = wav_np.mean(axis=1)
    if sr != 24000:
        wav_np = torchaudio.functional.resample(torch.from_numpy(wav_np).float(), sr, 24000).numpy()
    with torch.no_grad():
        emb = base_tts.model.extract_speaker_embedding(wav_np.astype(np.float32), 24000)
    return F.normalize(emb.float().cpu().flatten(), dim=0)

fp_vn = hf_hub_download("starrydark/Makise_Kurisu_Voice", "labels/kurisu_tts_v3/train.jsonl", repo_type="dataset")
vn_all = [json.loads(l) for l in open(fp_vn, encoding="utf-8")]
vn_strict = [
    r for r in vn_all
    if _is_clean_studio_vn(r)
    and r.get("v3_voice") == "kurisu"
    and r.get("v3_kept") is True
    and not r.get("v3_tag_only")
    and float(r.get("v3_amadeus_score") or 0.0) < 0.08
    and float(r.get("asr_cer_qwen3") or 0.0) <= 0.08
    and 12 <= len((r.get("v3_text") or "").strip()) <= 95
]
vn_embeds, calm_ref = [], None
for r in vn_strict[:24]:
    wp = hf_hub_download("starrydark/Makise_Kurisu_Voice", "data/train/" + r["file_name"], repo_type="dataset")
    y, sr = sf.read(wp, dtype="float32")
    if y.ndim > 1: y = y.mean(axis=1)
    y24 = torchaudio.functional.resample(torch.from_numpy(y).float(), sr, 24000).numpy() if sr != 24000 else y
    if len(vn_embeds) < 16:
        vn_embeds.append(ecapa_embed(y24, 24000))
    if calm_ref is None and 2.5 <= len(y24) / 24000.0 <= 5.5:
        ac_ref = measure_clip_8d(y24)
        if 195.0 <= ac_ref["f0_med"] <= 245.0:
            ref_path = f"{DATA_DIR}/ref_vn_studio.wav"
            sf.write(ref_path, y24, 24000, subtype="PCM_16")
            calm_ref = (ref_path, r["v3_text"].strip())
assert calm_ref is not None, "Nenhuma referencia calma encontrada na VN limpa!"
vn_centroid = F.normalize(torch.stack(vn_embeds).mean(dim=0), dim=0)
torch.save(vn_centroid, "/content/vn_centroid.pt")
print(f"VN estrita: {len(vn_strict)} clipes | Calm ref Zero-Shot: {calm_ref[0]} | BUCKET_A_STRATEGY={BUCKET_A_STRATEGY}")

# 2. Gerar áudios brutos do Bucket A conforme estratégia escolhida
tmp_src_dir = "/content/tmp_bucket_a_src"
os.makedirs(tmp_src_dir, exist_ok=True)
need_gen = [
    (i, mood, txt) for i, (mood, txt) in enumerate(KURISU_PTBR_LINES)
    if not os.path.exists(f"{DATA_DIR}/wavs_kurisu_ptbr/kpt_{i:03d}.wav")
    and not os.path.exists(f"{tmp_src_dir}/kpt_{i:03d}.wav")
]
if need_gen:
    if BUCKET_A_STRATEGY in ("base_zs_xvec", "base_zs_icl"):
        pr = base_tts.create_voice_clone_prompt(ref_audio=calm_ref[0], ref_text=calm_ref[1], x_vector_only_mode=(BUCKET_A_STRATEGY == "base_zs_xvec"))
        for i, _mood, txt in need_gen:
            wavs, sr = base_tts.generate_voice_clone(text=txt, language="Portuguese", voice_clone_prompt=pr, temperature=0.70, top_p=0.85, repetition_penalty=1.05, max_new_tokens=1024)
            sf.write(f"{tmp_src_dir}/kpt_{i:03d}.wav", wavs[0], sr, subtype="PCM_16")
    elif BUCKET_A_STRATEGY == "v3_raw":
        m_v3 = Qwen3TTSModel.from_pretrained(v3_dir, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="cuda:0")
        for i, _mood, txt in need_gen:
            wavs, sr = m_v3.generate_custom_voice(text=txt, speaker="kurisu", language="Portuguese", temperature=0.70, top_p=0.85, repetition_penalty=1.05, max_new_tokens=1024)
            sf.write(f"{tmp_src_dir}/kpt_{i:03d}.wav", wavs[0], sr, subtype="PCM_16")
        del m_v3; gc.collect(); torch.cuda.empty_cache()
    elif BUCKET_A_STRATEGY in ("applio_titan", "applio_ov2"):
        assert os.path.exists("/content/Applio/.venv/bin/python"), "Applio nao encontrado em /content/Applio"
        _drv = "/content/drive/MyDrive/OpenAmadeus/kurisu_rvc_applio"
        _pth = f"{_drv}/05_checkpoints_titan/weights/kurisu_applio_titan_100e_7700s.pth" if BUCKET_A_STRATEGY == "applio_titan" else f"{_drv}/04_checkpoints_ov2/weights/kurisu_applio_ov2_160e_12320s.pth"
        _idx = f"{_drv}/kurisu_rvc_applio.index"
        raw_dir = "/content/tmp_bucket_a_tts"
        os.makedirs(raw_dir, exist_ok=True)
        for i, mood, txt in need_gen:
            rate = {"lively": "+5%", "calm": "-4%", "neutral": "+0%"}[mood]
            await edge_tts.Communicate(txt, APPLIO_SOURCE_VOICE, rate=rate).save(f"{raw_dir}/kpt_{i:03d}.mp3")
        _batch_py = (
            "import glob, os, sys; sys.path.insert(0, '/content/Applio'); os.chdir('/content/Applio'); "
            "from rvc.infer.infer import VoiceConverter; vc = VoiceConverter(); "
            f"[vc.convert_audio(audio_input_path=p, audio_output_path=f'{tmp_src_dir}/{{os.path.splitext(os.path.basename(p))[0]}}.wav', model_path='{_pth}', index_path='{_idx}', pitch={APPLIO_PITCH}, f0_method='rmvpe', index_rate={APPLIO_INDEX_RATE}, volume_envelope=1.0, protect={APPLIO_PROTECT}, embedder_model='contentvec', export_format='WAV') for p in sorted(glob.glob('{raw_dir}/kpt_*.mp3'))]"
        )
        subprocess.run(["/content/Applio/.venv/bin/python", "-c", _batch_py], check=True)
        shutil.rmtree(raw_dir, ignore_errors=True)
    else:
        raise ValueError(f"BUCKET_A_STRATEGY invalida: {BUCKET_A_STRATEGY}")

# 3. Filtrar por portão acústico 8D + Whisper PT-BR (>= 0.90), SEMPRE com train_code_predictor=False
w_asr = WhisperModel("small", device="cpu", compute_type="int8")
def norm_text(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-zà-öø-ÿ0-9\s]", "", s.lower())).strip()

kept_a, rej_counts = [], collections.Counter()
for i, (_mood, text) in enumerate(KURISU_PTBR_LINES):
    out_wav = f"{DATA_DIR}/wavs_kurisu_ptbr/kpt_{i:03d}.wav"
    if os.path.exists(out_wav):
        y24, _ = sf.read(out_wav, dtype="float32")
        kept_a.append({"audio": out_wav, "text": text, "language": "Portuguese", "speaker": "kurisu", "bucket": "kurisu_ptbr", "train_code_predictor": False, "dur": len(y24) / 24000.0})
        continue
    src_wav = f"{tmp_src_dir}/kpt_{i:03d}.wav"
    if not os.path.exists(src_wav):
        rej_counts["missing"] += 1; continue
    y, sr = sf.read(src_wav, dtype="float32")
    if y.ndim > 1: y = y.mean(axis=1)
    y24 = torchaudio.functional.resample(torch.from_numpy(y).float(), sr, 24000).numpy() if sr != 24000 else y
    dur, rms = len(y24) / 24000.0, float(np.sqrt(np.mean(y24 ** 2)))
    if dur < 1.0 or dur > 11.0 or rms < 0.01:
        rej_counts["dur_rms"] += 1; continue
    if APPLY_DATA_WARMTH_EQ:
        y24 = apply_kurisu_warmth_eq(y24, 24000)
    sim = float((ecapa_embed(y24, 24000) * vn_centroid).sum())
    ac = measure_clip_8d(y24)
    if sim < 0.945: rej_counts["ecapa"] += 1; continue
    if ac["hnr"] < 10.0: rej_counts["hnr"] += 1; continue
    if not (195.0 <= ac["f0_med"] <= 315.0): rej_counts["f0"] += 1; continue
    if not (1050.0 <= ac["f2"] <= 1450.0): rej_counts["f2"] += 1; continue
    if ac["low_hi_db"] < 10.0: rej_counts["low_hi"] += 1; continue
    y16 = torchaudio.functional.resample(torch.from_numpy(y24).float(), 24000, 16000).numpy().astype(np.float32)
    segs, _ = w_asr.transcribe(y16, beam_size=3, language="pt")
    seq_ratio = difflib.SequenceMatcher(None, norm_text(text), norm_text(" ".join(s.text.strip() for s in segs))).ratio()
    if seq_ratio < WHISPER_MIN_RATIO:
        rej_counts["asr"] += 1; continue
    sf.write(out_wav, y24, 24000, subtype="PCM_16")
    kept_a.append({
        "audio": out_wav, "text": text, "language": "Portuguese", "speaker": "kurisu",
        "bucket": "kurisu_ptbr", "train_code_predictor": False, "src": BUCKET_A_STRATEGY,
        "ecapa_vn": sim, "f0_med": ac["f0_med"], "hnr": ac["hnr"], "f1": ac["f1"], "f2": ac["f2"], "f3": ac["f3"], "low_hi_db": ac["low_hi_db"], "h1_h2": ac["h1_h2"],
        "dur": dur, "seq_ratio": seq_ratio,
    })

shutil.rmtree(tmp_src_dir, ignore_errors=True)
with open(f"{DATA_DIR}/bucket_a.jsonl", "w", encoding="utf-8") as f:
    for r in kept_a:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Bucket A v2 ({BUCKET_A_STRATEGY}): {len(kept_a)} clipes ({sum(r['dur'] for r in kept_a)/60:.1f} min) | rejeicoes={dict(rej_counts)}")
if kept_a and "f2" in kept_a[0]:
    print("  Medianas Bucket A:",
          f"HNR={np.median([r['hnr'] for r in kept_a if 'hnr' in r]):.2f}",
          f"F0={np.median([r['f0_med'] for r in kept_a if 'f0_med' in r]):.1f}Hz",
          f"F1/2/3={np.median([r['f1'] for r in kept_a if 'f1' in r]):.0f}/{np.median([r['f2'] for r in kept_a if 'f2' in r]):.0f}/{np.median([r['f3'] for r in kept_a if 'f3' in r]):.0f}",
          f"Low/Hi={np.median([r['low_hi_db'] for r in kept_a if 'low_hi_db' in r]):+.1f}dB",
          f"H12={np.median([r['h1_h2'] for r in kept_a if 'h1_h2' in r]):+.1f}dB",
          f"ECAPA={np.median([r['ecapa_vn'] for r in kept_a if 'ecapa_vn' in r]):.4f}")
del base_tts, w_asr
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# PASSO 3: Buckets B (BRSpeech Nativo Multi-Locutor) + C (VN Limpa v2) + Tokens 12Hz
# ==============================================================================
import os, io, re, gc, glob, json, collections, concurrent.futures
import numpy as np
import soundfile as sf
import torch
import torch.nn.functional as F
import torchaudio
import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download
from safetensors import safe_open
from qwen_tts import Qwen3TTSModel

DATA_DIR = "/content/ptbr_kurisu_data_v2"
os.makedirs(f"{DATA_DIR}/wavs_brspeech", exist_ok=True)
os.makedirs(f"{DATA_DIR}/wavs_vn_clean", exist_ok=True)

v3_dir = sorted(glob.glob("/root/.cache/huggingface/hub/models--starrydark--Makise_Kurisu_Qwen3_TTS/snapshots/*"))[-1]
base_dir = sorted(glob.glob("/root/.cache/huggingface/hub/models--Qwen--Qwen3-TTS-12Hz-1.7B-Base/snapshots/*"))[-1]

with safe_open(os.path.join(v3_dir, "model.safetensors"), framework="pt") as f:
    v3_spk_rows = f.get_tensor("talker.model.codec_embedding.weight")[3000:3003].clone().float()
TARGET_SPK_NORM = float(v3_spk_rows[0].norm())

base_tts = Qwen3TTSModel.from_pretrained(base_dir, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="cuda:0")
def extract_raw_ecapa(y24):
    with torch.no_grad():
        emb = base_tts.model.extract_speaker_embedding(y24.astype(np.float32), 24000)
    return F.normalize(emb.float().cpu().flatten(), dim=0)

# Bucket B: BRSpeech-TTS multi-speaker brasileiro nativo
shards = [
    "data/test-00000-of-00002.parquet",
    "data/test-00001-of-00002.parquet",
    "data/train-00000-of-00053.parquet",
    "data/train-00005-of-00053.parquet",
    "data/train-00015-of-00053.parquet",
]
per_spk_count = collections.Counter()
per_spk_embeds = collections.defaultdict(list)
rows_b = []

for shard in shards:
    if len(rows_b) >= 300:
        break
    pf = hf_hub_download("freds0/BRSpeech-TTS", shard, repo_type="dataset")
    t = pq.read_table(pf)
    for i in range(t.num_rows):
        if len(rows_b) >= 300:
            break
        if t["accent"][i].as_py() != "brazilian":
            continue
        lev = float(t["levenshtein"][i].as_py() or 0.0)
        dur = float(t["duration"][i].as_py() or 0.0)
        txt = (t["text"][i].as_py() or "").strip()
        spk = str(t["speaker"][i].as_py())
        if lev < 0.94 or dur < 1.5 or dur > 8.5 or len(txt) < 15 or per_spk_count[spk] >= 22:
            continue
        y, sr = sf.read(io.BytesIO(t["audio"][i].as_py()["bytes"]), dtype="float32")
        if y.ndim > 1:
            y = y.mean(axis=1)
        if float(np.sqrt(np.mean(y ** 2))) < 0.01:
            continue
        y24 = torchaudio.functional.resample(torch.from_numpy(y).float(), sr, 24000).numpy() if sr != 24000 else y
        spk_key = f"brs_{spk}"
        per_spk_embeds[spk_key].append(extract_raw_ecapa(y24))
        per_spk_count[spk] += 1
        out_wav = f"{DATA_DIR}/wavs_brspeech/{spk_key}_{len(rows_b):03d}.wav"
        sf.write(out_wav, y24, 24000, subtype="PCM_16")
        rows_b.append({"audio": out_wav, "text": txt, "language": "Portuguese", "speaker": spk_key, "bucket": "brspeech_native", "train_code_predictor": False, "dur": len(y24) / 24000.0})

spk_table = {"kurisu": v3_spk_rows[0], "amadeus": v3_spk_rows[1], "phone": v3_spk_rows[2]}
for spk_key, embs in per_spk_embeds.items():
    spk_table[spk_key] = F.normalize(torch.stack(embs).mean(dim=0), dim=0) * TARGET_SPK_NORM
torch.save(spk_table, f"{DATA_DIR}/spk_table.pt")
with open(f"{DATA_DIR}/bucket_b.jsonl", "w", encoding="utf-8") as f:
    for r in rows_b:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
del base_tts
gc.collect(); torch.cuda.empty_cache()
print(f"Bucket B pronto: {len(rows_b)} clipes em {len(per_spk_embeds)} locutores brasileiros")

# Bucket C v2: Âncora de estúdio limpa VN (apenas SG/PHENO idx >= 179, sem Amadeus/auditório)
def _is_clean_studio_vn(r):
    fn = os.path.basename(str(r.get("file_name") or ""))
    if not fn.startswith(("game_ja_SG_kurisu_", "game_ja_PHENO_kurisu_")):
        return False
    m = re.search(r"kurisu_(\d+)", fn)
    return bool(m and int(m.group(1)) >= 179)

def _median_f0_autocorr(y, sr=24000):
    win, hop = 960, 480
    lo, hi = sr // 380, sr // 140
    vals = []
    for s in range(0, len(y) - win, hop):
        seg = y[s : s + win] - np.mean(y[s : s + win])
        if np.sqrt(np.mean(seg ** 2)) < 0.015:
            continue
        acf = np.correlate(seg, seg, mode="full")[win - 1 :]
        r0 = float(acf[0])
        if r0 <= 1e-9:
            continue
        sub = acf[lo : hi + 1]
        idx = int(np.argmax(sub))
        if sub[idx] / r0 > 0.45:
            vals.append(sr / (lo + idx))
    return float(np.median(vals)) if len(vals) >= 3 else 0.0

fp_vn = hf_hub_download("starrydark/Makise_Kurisu_Voice", "labels/kurisu_tts_v3/train.jsonl", repo_type="dataset")
vn_all = [json.loads(l) for l in open(fp_vn, encoding="utf-8")]
strict = [
    r for r in vn_all
    if _is_clean_studio_vn(r)
    and r.get("v3_voice") == "kurisu" and r.get("v3_kept") is True and not r.get("v3_tag_only")
    and float(r.get("v3_amadeus_score") or 0.0) < 0.08 and float(r.get("asr_cer_qwen3") or 0.0) <= 0.08
    and 12 <= len((r.get("v3_text") or "").strip()) <= 95
]
rng = np.random.default_rng(42)
sel_idx = rng.choice(len(strict), size=min(260, len(strict)), replace=False)

def fetch_vn(idx):
    r = strict[int(idx)]
    lp = hf_hub_download("starrydark/Makise_Kurisu_Voice", "data/train/" + r["file_name"], repo_type="dataset")
    y, sr = sf.read(lp, dtype="float32")
    if y.ndim > 1:
        y = y.mean(axis=1)
    y24 = torchaudio.functional.resample(torch.from_numpy(y).float(), sr, 24000).numpy() if sr != 24000 else y
    dur = len(y24) / 24000.0
    if dur < 1.0 or dur > 9.5 or float(np.sqrt(np.mean(y24 ** 2))) < 0.008:
        return None
    f0_med = _median_f0_autocorr(y24, 24000)
    if not (205.0 <= f0_med <= 285.0):
        return None
    out_wav = f"{DATA_DIR}/wavs_vn_clean/{os.path.basename(r['file_name'])}"
    if not out_wav.endswith(".wav"):
        out_wav += ".wav"
    sf.write(out_wav, y24, 24000, subtype="PCM_16")
    return {"audio": out_wav, "text": r["v3_text"].strip(), "language": "Japanese", "speaker": "kurisu", "bucket": "vn_japanese", "train_code_predictor": True, "f0_med": f0_med, "dur": dur}

with concurrent.futures.ThreadPoolExecutor(max_workers=12) as ex:
    rows_c = [res for res in ex.map(fetch_vn, sel_idx) if res is not None]
with open(f"{DATA_DIR}/bucket_c.jsonl", "w", encoding="utf-8") as f:
    for r in rows_c:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"Bucket C pronto: {len(rows_c)} clipes limpos de estúdio da VN (estrito)")

# Tokenização acústica 12Hz
all_rows = []
for fn in ["bucket_a.jsonl", "bucket_b.jsonl", "bucket_c.jsonl"]:
    all_rows.extend(json.loads(l) for l in open(f"{DATA_DIR}/{fn}", encoding="utf-8"))

m = Qwen3TTSModel.from_pretrained(v3_dir, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="cuda:0")
encoded_items = []
for s in range(0, len(all_rows), 16):
    chunk = all_rows[s : s + 16]
    enc = m.model.speech_tokenizer.encode([r["audio"] for r in chunk])
    for r, codes in zip(chunk, enc.audio_codes):
        c_cpu = codes.detach().cpu().to(torch.int32)
        if c_cpu.shape[0] < 10 or c_cpu.shape[0] > 200 or c_cpu.shape[1] != 16:
            continue
        t_str = f"<|im_start|>assistant\n{r['text']}<|im_end|>\n<|im_start|>assistant\n"
        t_ids = m.processor(text=t_str, return_tensors="pt")["input_ids"].squeeze(0).to(torch.int32)
        if t_ids.shape[0] <= 8:
            continue
        # Isolamento estrito: apenas Bucket C (vn_japanese) pode treinar codebooks 1..15
        is_vn = (r["bucket"] == "vn_japanese")
        encoded_items.append({"input_ids": t_ids, "audio_codes": c_cpu, "language": r["language"], "speaker": r["speaker"], "bucket": r["bucket"], "train_code_predictor": is_vn, "text": r["text"], "audio": r["audio"]})

gen = torch.Generator().manual_seed(2026)
train_items, val_items = [], []
for b_name in ["kurisu_ptbr", "brspeech_native", "vn_japanese"]:
    sub = [x for x in encoded_items if x["bucket"] == b_name]
    for pos, iv in enumerate(torch.randperm(len(sub), generator=gen).tolist()):
        (val_items if pos < 10 else train_items).append(sub[iv])

cp_counts = collections.Counter(x["bucket"] for x in train_items if x["train_code_predictor"])
torch.save({"train": train_items, "val": val_items}, f"{DATA_DIR}/encoded_dataset.pt")
print(f"Dataset 12Hz pronto: train={len(train_items)}, val={len(val_items)} | train_code_predictor=True apenas em: {dict(cp_counts)}")
del m
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# PASSO 4: Treino LoRA v2 Seletivo (Talker + text_projection; CodePredictor Isolado)
# ==============================================================================
import os, gc, glob, math, time, shutil, collections
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file, save_file
from qwen_tts import Qwen3TTSModel

DATA_DIR = "/content/ptbr_kurisu_data_v2"
CKPT_DIR = "/content/kurisu_ptbr_v2_lora_ckpts"
MERGED_DIR = "/content/kurisu_ptbr_v3_v2_merged"
WARM_START_EP6 = False  # False por padrao para nao herdar vicio de F2/F3 agudo da v1 (ep6)
CODE_PREDICTOR_MODE = "frozen"  # "frozen" (0 gradientes no timbre) | "vn_only" (somente Bucket C)
PREV_LORA_CKPTS = [
    "/content/kurisu_ptbr_lora_ckpts/lora_ep6.pt",
    "/content/drive/MyDrive/OpenAmadeus/kurisu_ptbr_extras/kurisu_ptbr_lora_ckpts/lora_ep6.pt",
    "/content/drive/MyDrive/OpenAmadeus/kurisu_ptbr_extras/lora_ep6.pt",
]
os.makedirs(CKPT_DIR, exist_ok=True)

ds = torch.load(f"{DATA_DIR}/encoded_dataset.pt", weights_only=False)
train_items, val_items = ds["train"], ds["val"]
spk_table = torch.load(f"{DATA_DIR}/spk_table.pt", weights_only=False)
v3_dir = sorted(glob.glob("/root/.cache/huggingface/hub/models--starrydark--Makise_Kurisu_Qwen3_TTS/snapshots/*"))[-1]

tts = Qwen3TTSModel.from_pretrained(v3_dir, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="cuda:0")
model = tts.model
model.train()
tc, cfg = model.config.talker_config, model.config
orig_spk_rows = model.talker.model.codec_embedding.weight[3000:3003].detach().cpu().clone()

class LoRALinear(nn.Module):
    def __init__(self, base: nn.Linear, r: int = 32, alpha: float = 64.0, dropout: float = 0.05):
        super().__init__()
        self.base = base
        for p in self.base.parameters():
            p.requires_grad_(False)
        self.scaling = alpha / r
        self.drop = nn.Dropout(dropout) if dropout > 0 else nn.Identity()
        dev = base.weight.device
        self.lora_A = nn.Parameter(torch.empty(r, base.in_features, device=dev, dtype=torch.float32))
        self.lora_B = nn.Parameter(torch.zeros(base.out_features, r, device=dev, dtype=torch.float32))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))

    def forward(self, x):
        out = self.base(x)
        delta = F.linear(F.linear(self.drop(x.float()), self.lora_A), self.lora_B) * self.scaling
        return out + delta.to(out.dtype)

for p in model.parameters():
    p.requires_grad_(False)

TARGET_LEAVES = {"q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "linear_fc1", "linear_fc2", "small_to_mtp_projection"}
injected = collections.Counter()
def inject_lora(parent):
    for name, child in list(parent.named_children()):
        if name == "code_predictor" and CODE_PREDICTOR_MODE == "frozen":
            continue
        if isinstance(child, nn.Linear) and name in TARGET_LEAVES:
            setattr(parent, name, LoRALinear(child, r=32, alpha=64.0, dropout=0.05))
            injected[name] += 1
        else:
            inject_lora(child)

inject_lora(model.talker)
warm_loaded = 0
if WARM_START_EP6:
    prev_ckpt = next((p for p in PREV_LORA_CKPTS if os.path.exists(p)), None)
    if not prev_ckpt and os.path.exists("/content/drive/MyDrive/OpenAmadeus"):
        found = sorted(glob.glob("/content/drive/MyDrive/OpenAmadeus/**/lora_ep6.pt", recursive=True))
        prev_ckpt = found[0] if found else None
    if prev_ckpt:
        prev_sd = torch.load(prev_ckpt, map_location="cpu", weights_only=True)
        cur_params = dict(model.named_parameters())
        for k, v in prev_sd.items():
            if k.startswith("talker.code_predictor."):
                continue
            if k in cur_params and cur_params[k].shape == v.shape:
                cur_params[k].data.copy_(v.to(device=cur_params[k].device, dtype=cur_params[k].dtype))
                warm_loaded += 1
        print(f"Warm-start ep6: {warm_loaded} tensores Talker/text_proj carregados (CodePredictor isolado)")

lora_params = [p for p in model.parameters() if p.requires_grad]
cp_lora_cnt = sum(1 for n, _ in model.named_parameters() if n.startswith("talker.code_predictor.") and ".lora_" in n)
print(f"CODE_PREDICTOR_MODE={CODE_PREDICTOR_MODE} | WARM_START_EP6={WARM_START_EP6} | LoRA={sum(injected.values())} matrizes ({sum(p.numel() for p in lora_params)/1e6:.2f}M params) | cp_lora={cp_lora_cnt}")

spk_gpu = {k: v.to(device="cuda:0", dtype=torch.bfloat16) for k, v in spk_table.items()}
special_txt = torch.tensor([[cfg.tts_bos_token_id, cfg.tts_eos_token_id, cfg.tts_pad_token_id]], device="cuda:0", dtype=torch.long)
c_pad_1 = torch.tensor([[tc.codec_pad_id]], device="cuda:0", dtype=torch.long)
c_bos_1 = torch.tensor([[tc.codec_bos_id]], device="cuda:0", dtype=torch.long)
c_eos_1 = torch.tensor([[tc.codec_eos_token_id]], device="cuda:0", dtype=torch.long)

def build_item(item, p_auto=0.15, rng_gen=None):
    dev = "cuda:0"
    t_ids = item["input_ids"].unsqueeze(0).to(dev, dtype=torch.long)
    codes = item["audio_codes"].to(dev, dtype=torch.long)
    T_txt, T_aud = t_ids.shape[1] - 8, codes.shape[0]
    spk_vec = spk_gpu[item["speaker"]].view(1, 1, -1)
    use_auto = (item["speaker"] == "kurisu") and (rng_gen is not None) and (torch.rand((), generator=rng_gen).item() < p_auto)

    tts_bos_e, tts_eos_e, tts_pad_e = model.talker.text_projection(model.talker.get_text_embeddings()(special_txt)).chunk(3, dim=1)
    role_e = model.talker.text_projection(model.talker.get_text_embeddings()(t_ids[:, :3]))
    if use_auto:
        c0 = model.talker.get_input_embeddings()(torch.tensor([[tc.codec_nothink_id, tc.codec_think_bos_id, tc.codec_think_eos_id]], device=dev))
        prefill_e = torch.cat([tts_pad_e.expand(1, 4, -1), tts_bos_e], dim=1) + torch.cat([c0, spk_vec, model.talker.get_input_embeddings()(c_pad_1)], dim=1)
    else:
        lid = tc.codec_language_id[item["language"].lower()]
        c0 = model.talker.get_input_embeddings()(torch.tensor([[tc.codec_think_id, tc.codec_think_bos_id, lid, tc.codec_think_eos_id]], device=dev))
        prefill_e = torch.cat([tts_pad_e.expand(1, 5, -1), tts_bos_e], dim=1) + torch.cat([c0, spk_vec, model.talker.get_input_embeddings()(c_pad_1)], dim=1)

    txt_body_e = torch.cat([model.talker.text_projection(model.talker.get_text_embeddings()(t_ids[:, 3:-5])), tts_eos_e], dim=1)
    txt_block_e = txt_body_e + model.talker.get_input_embeddings()(torch.full((1, T_txt + 1), tc.codec_pad_id, device=dev))
    bos_e = tts_pad_e + model.talker.get_input_embeddings()(c_bos_1)
    frame_e = model.talker.get_input_embeddings()(codes[:, 0].unsqueeze(0))
    for i in range(1, tc.num_code_groups):
        frame_e = frame_e + model.talker.code_predictor.get_input_embeddings()[i - 1](codes[:, i].unsqueeze(0))
    aud_in_e = frame_e + tts_pad_e
    eos_in_e = tts_pad_e + model.talker.get_input_embeddings()(c_eos_1)

    seq_e = torch.cat([role_e, prefill_e, txt_block_e, bos_e, aud_in_e, eos_in_e], dim=1).squeeze(0)
    L = seq_e.shape[0]
    P = 3 + prefill_e.shape[1] + (T_txt + 1)
    labels = torch.full((L,), -100, dtype=torch.long, device=dev)
    labels[P + 1 : P + 1 + T_aud] = codes[:, 0]
    labels[P + 1 + T_aud] = tc.codec_eos_token_id
    codec_mask = torch.zeros((L,), dtype=torch.bool, device=dev)
    codec_mask[P + 1 : P + 1 + T_aud] = True
    train_cp = (item.get("bucket") == "vn_japanese") and bool(item.get("train_code_predictor", False)) and (CODE_PREDICTOR_MODE == "vn_only")
    return seq_e, labels, codec_mask, codes, train_cp

def compute_batch_loss(batch_items, p_auto=0.15, rng_gen=None, sub_weight=0.30):
    built = [build_item(it, p_auto=p_auto, rng_gen=rng_gen) for it in batch_items]
    B, L_max, D = len(built), max(s[0].shape[0] for s in built), built[0][0].shape[1]
    inputs_embeds = torch.zeros((B, L_max, D), dtype=built[0][0].dtype, device="cuda:0")
    attn_mask = torch.zeros((B, L_max), dtype=torch.long, device="cuda:0")
    labels = torch.full((B, L_max), -100, dtype=torch.long, device="cuda:0")
    codec_mask = torch.zeros((B, L_max), dtype=torch.bool, device="cuda:0")
    for b, (se, la, ma, _, _) in enumerate(built):
        l = se.shape[0]
        inputs_embeds[b, :l], attn_mask[b, :l], labels[b, :l], codec_mask[b, :l] = se, 1, la, ma

    cap = {}
    hook = model.talker.model.register_forward_hook(lambda _m, _i, o: cap.update({"h": o.last_hidden_state}))
    try:
        out = model.talker(inputs_embeds=inputs_embeds, attention_mask=attn_mask, labels=labels, use_cache=False)
    finally:
        hook.remove()

    sel_h = cap["h"][:, :-1][codec_mask[:, 1:]]
    sel_codes = torch.cat([x[3] for x in built], dim=0)
    cp_mask = torch.cat([
        torch.full((x[3].shape[0],), bool(x[4]), dtype=torch.bool, device="cuda:0")
        for x in built
    ], dim=0)
    if CODE_PREDICTOR_MODE == "vn_only" and sub_weight > 0 and cp_mask.any():
        cp_h = sel_h[cp_mask]
        cp_codes = sel_codes[cp_mask]
        sub_in = [cp_h.unsqueeze(1), model.talker.get_input_embeddings()(cp_codes[:, :1])]
        for i in range(1, tc.num_code_groups - 1):
            sub_in.append(model.talker.code_predictor.get_input_embeddings()[i - 1](cp_codes[:, i : i + 1]))
        sub_logits = model.talker.code_predictor.forward_finetune(inputs_embeds=torch.cat(sub_in, dim=1), labels=None).logits
        sub_loss = F.cross_entropy(sub_logits.float().reshape(-1, sub_logits.size(-1)), cp_codes[:, 1:].reshape(-1), ignore_index=-100)
        total_loss = out.loss.float() + sub_weight * sub_loss
        sub_val = float(sub_loss.detach())
    else:
        total_loss = out.loss.float()
        sub_val = 0.0
    return total_loss, float(out.loss.detach()), sub_val

SUB_WEIGHT = 0.30 if CODE_PREDICTOR_MODE == "vn_only" else 0.0
EPOCHS, BS, GRAD_ACCUM, LR = (4 if warm_loaded > 0 else 6), 4, 4, (6e-5 if warm_loaded > 0 else 8e-5)
total_opt = EPOCHS * math.ceil(math.ceil(len(train_items) / BS) / GRAD_ACCUM)
warmup = max(8, int(0.06 * total_opt))
opt = torch.optim.AdamW(lora_params, lr=LR, betas=(0.9, 0.98), weight_decay=0.01)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warmup if s < warmup else 0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * min(1.0, (s - warmup) / max(1, total_opt - warmup)))))
rng = torch.Generator().manual_seed(42)

log_lines = []
for ep in range(1, EPOCHS + 1):
    t_ep = time.perf_counter()
    model.train()
    perm = torch.randperm(len(train_items), generator=rng).tolist()
    shuffled = [train_items[i] for i in perm]
    batches = [shuffled[i : i + BS] for i in range(0, len(shuffled), BS)]
    opt.zero_grad(set_to_none=True)
    ep_tot, ep_tk, ep_sb = [], [], []
    for bi, batch in enumerate(batches):
        loss, tk, sb = compute_batch_loss(batch, p_auto=0.15, rng_gen=rng, sub_weight=SUB_WEIGHT)
        (loss / GRAD_ACCUM).backward()
        ep_tot.append(float(loss.detach())); ep_tk.append(tk); ep_sb.append(sb)
        if (bi + 1) % GRAD_ACCUM == 0 or (bi + 1) == len(batches):
            torch.nn.utils.clip_grad_norm_(lora_params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
    model.eval()
    val_by_b = collections.defaultdict(list)
    with torch.no_grad():
        for vit in val_items:
            _, vtk, _ = compute_batch_loss([vit], p_auto=0.0, rng_gen=None, sub_weight=0.0)
            val_by_b[vit["bucket"]].append(vtk)
    val_str = " ".join(f"{k[:3]}={sum(v)/len(v):.3f}" for k, v in sorted(val_by_b.items()))
    if ep in (2, 4, 6) or ep == EPOCHS:
        torch.save({k: v.detach().cpu().clone() for k, v in model.state_dict().items() if ".lora_A" in k or ".lora_B" in k}, f"{CKPT_DIR}/lora_ep{ep}.pt")
    msg = f"Ep {ep}/{EPOCHS} ({time.perf_counter()-t_ep:.1f}s) | loss={sum(ep_tot)/len(ep_tot):.4f} talker={sum(ep_tk)/len(ep_tk):.4f} sub={sum(ep_sb)/len(ep_sb):.4f} | val[{val_str}]"
    print(msg, flush=True); log_lines.append(msg)

open("/content/train_log_v2.txt", "w").write("\n".join(log_lines) + "\n")
assert float((model.talker.model.codec_embedding.weight[3000:3003].detach().cpu() - orig_spk_rows).abs().max()) == 0.0
del tts, model, opt
gc.collect(); torch.cuda.empty_cache()

# Merge do checkpoint final (EPOCHS) para formato HF compatível com FasterQwen3TTS
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
shutil.copytree(v3_dir, MERGED_DIR, ignore=lambda src, names: ["model.safetensors"] if os.path.abspath(src) == os.path.abspath(v3_dir) else [])
sd = load_file(os.path.join(v3_dir, "model.safetensors"))
lora_sd = torch.load(f"{CKPT_DIR}/lora_ep{EPOCHS}.pt", map_location="cpu", weights_only=True)
for mod in sorted({k.rsplit(".", 1)[0] for k in lora_sd if k.endswith(".lora_A")}):
    wk = f"{mod}.weight"
    A, B = lora_sd[f"{mod}.lora_A"].float(), lora_sd[f"{mod}.lora_B"].float()
    sd[wk] = (sd[wk].float() + 2.0 * (B @ A)).to(sd[wk].dtype)
save_file(sd, os.path.join(MERGED_DIR, "model.safetensors"))
print(f"Modelo PT-BR mesclado pronto em: {MERGED_DIR}")

In [ ]:
# ==============================================================================
# PASSO 5: Validacao Objetiva v2 (FasterQwen3TTS + ECAPA + Whisper PT + 8D F2/LowHi)
# ==============================================================================
import os, gc, re, glob, time, difflib, inspect
import numpy as np
import soundfile as sf
import scipy.signal as sig
from scipy.linalg import solve_toeplitz
import torch
import torch.nn.functional as F
import torchaudio
from qwen_tts import Qwen3TTSModel
from faster_whisper import WhisperModel

def _patch_static_layer():
    try:
        from transformers.cache_utils import StaticLayer
        if not hasattr(StaticLayer, "_patched_by_openamadeus"):
            orig_fn = StaticLayer.lazy_initialization
            if len(inspect.signature(orig_fn).parameters) == 2:
                def _compat(self, key_states, value_states=None):
                    return orig_fn(self, key_states)
                StaticLayer.lazy_initialization = _compat
                StaticLayer._patched_by_openamadeus = True
    except Exception:
        pass

_patch_static_layer()
from faster_qwen3_tts import FasterQwen3TTS

MERGED_DIR = "/content/kurisu_ptbr_v3_v2_merged" if os.path.exists("/content/kurisu_ptbr_v3_v2_merged") else "/content/kurisu_ptbr_v3_merged"
base_dir = sorted(glob.glob("/root/.cache/huggingface/hub/models--Qwen--Qwen3-TTS-12Hz-1.7B-Base/snapshots/*"))[-1]
vn_centroid = torch.load("/content/vn_centroid.pt")

fast_tts = FasterQwen3TTS.from_pretrained(MERGED_DIR, device="cuda:0", dtype=torch.bfloat16, max_seq_len=2048)
if getattr(fast_tts, "predictor_graph", None) is not None:
    fast_tts.predictor_graph.temperature = 0.6
    fast_tts.predictor_graph.top_p = 0.9
fast_tts.warmup(prefill_len=100)
for _ in fast_tts.generate_custom_voice_streaming(text="Oi.", speaker="kurisu", language="Portuguese", instruct=None, chunk_size=8):
    pass

TESTS = [
    ("pt_ood1", "Portuguese", "Ei, presta atenção! O sinal elétrico do hipocampo estabilizou agora há pouco, viu?"),
    ("pt_ood2", "Portuguese", "Calma, respira fundo! Mesmo se o servidor cair de madrugada, nós vamos recuperar os arquivos amanhã."),
    ("pt_ood3", "Portuguese", "Que código bem estruturado, admito que fiquei positivamente surpresa com esse resultado."),
    ("pt_short", "Portuguese", "Oi! Tudo certo por aí? Já tomou seu café hoje?"),
    ("ja_reg",  "Japanese",   "それくらいは分かるみたいね……牧瀬紅莉栖？"),
]

os.makedirs("/content/eval_ptbr_final", exist_ok=True)
outs = []
for key, lang, txt in TESTS:
    t0, ttfa, sr_out, chunks = time.perf_counter(), None, 24000, []
    for c, sr, _ in fast_tts.generate_custom_voice_streaming(text=txt, speaker="kurisu", language=lang, temperature=0.70, top_p=0.85, repetition_penalty=1.05, chunk_size=8):
        if ttfa is None:
            ttfa = (time.perf_counter() - t0) * 1000
        sr_out = sr
        chunks.append(np.asarray(c, dtype=np.float32).flatten())
    y = np.concatenate(chunks)
    sf.write(f"/content/eval_ptbr_final/{key}.wav", y, sr_out, subtype="PCM_16")
    outs.append((key, y, sr_out, txt, ttfa or 0.0, (len(y) / sr_out) / max(0.01, time.perf_counter() - t0)))

del fast_tts
gc.collect(); torch.cuda.empty_cache()

def _measure_8d(y24):
    y_n = y24 / (float(np.max(np.abs(y24))) + 1e-9) * 0.89
    y16 = sig.resample_poly(y_n, 2, 3).astype(np.float32)
    win, hop = 640, 160
    hann = np.hanning(win)
    ac_w = np.maximum(np.correlate(hann, hann, mode="full")[win - 1 :] / max(1e-9, float(np.sum(hann ** 2))), 1e-6)
    min_lag, max_lag = 16000 // 550, 16000 // 110
    f0s, hnrs, h1h2s = [], [], []
    for s in range(0, len(y16) - win, hop):
        seg = y16[s : s + win] - float(np.mean(y16[s : s + win]))
        if np.sqrt(np.mean(seg ** 2)) < 0.03:
            continue
        wseg = seg * hann
        ac = np.correlate(wseg, wseg, mode="full")[win - 1 :]
        if ac[0] <= 1e-9:
            continue
        ac_n = (ac / ac[0]) / ac_w
        sl = ac_n[min_lag : max_lag + 1]
        k = int(np.argmax(sl))
        r_max = float(sl[k])
        if r_max > 0.35:
            f0 = 16000.0 / (min_lag + k)
            f0s.append(f0)
            r_c = min(max(r_max, 0.01), 0.998)
            hnrs.append(10.0 * np.log10(r_c / (1.0 - r_c)))
            sp = np.abs(np.fft.rfft(wseg, n=2048)) + 1e-9
            fr = np.fft.rfftfreq(2048, 1.0 / 16000)
            h1 = float(np.max(sp[(fr >= f0 * 0.8) & (fr <= f0 * 1.25)]))
            h2 = float(np.max(sp[(fr >= f0 * 1.7) & (fr <= f0 * 2.3)]))
            h1h2s.append(20.0 * np.log10(h1 / h2))
    y12 = sig.resample_poly(y_n, 1, 2).astype(np.float32)
    win12, hop12, ham = 600, 240, np.hamming(600)
    f1s, f2s, f3s = [], [], []
    for s in range(0, len(y12) - win12, hop12):
        seg = y12[s : s + win12]
        if np.sqrt(np.mean(seg ** 2)) < 0.04:
            continue
        seg = np.append(seg[0], seg[1:] - 0.97 * seg[:-1]) * ham
        r = np.correlate(seg, seg, mode="full")[win12 - 1 : win12 + 14]
        try:
            a = solve_toeplitz((r[:-1], r[:-1]), -r[1:])
            roots = np.roots(np.concatenate([[1.0], a]))
            roots = roots[np.imag(roots) >= 0.01]
            angs = np.arctan2(np.imag(roots), np.real(roots)) * (12000.0 / (2.0 * np.pi))
            bws = -0.5 * (12000.0 / (2.0 * np.pi)) * np.log(np.abs(roots) + 1e-9)
            valid = sorted(float(fr) for fr, bw in zip(angs, bws) if 220.0 < fr < 4800.0 and bw < 500.0)
            if len(valid) >= 3:
                f1s.append(valid[0]); f2s.append(valid[1]); f3s.append(valid[2])
        except Exception:
            pass
    y_rms = y_n / (float(np.sqrt(np.mean(y_n ** 2))) + 1e-9)
    freqs, pxx = sig.welch(y_rms, fs=24000, nperseg=2048)
    e_low = float(np.sum(pxx[(freqs >= 120) & (freqs <= 700)])) + 1e-12
    e_hi = float(np.sum(pxx[(freqs >= 1800) & (freqs <= 5500)])) + 1e-12
    return {
        "hnr": float(np.median(hnrs)) if hnrs else 0.0,
        "f0": float(np.median(f0s)) if f0s else 0.0,
        "f1": float(np.median(f1s)) if f1s else 0.0,
        "f2": float(np.median(f2s)) if f2s else 0.0,
        "f3": float(np.median(f3s)) if f3s else 0.0,
        "low_hi": float(10.0 * np.log10(e_low / e_hi)),
        "h1_h2": float(np.median(h1h2s)) if h1h2s else 0.0,
    }

def _biquad_eval(f0, gain_db, q, fs=24000):
    a_amp = 10.0 ** (gain_db / 40.0)
    w0 = 2.0 * np.pi * f0 / fs
    alpha = np.sin(w0) / (2.0 * q)
    cw = np.cos(w0)
    b0, b1, b2 = 1.0 + alpha * a_amp, -2.0 * cw, 1.0 - alpha * a_amp
    a0, a1, a2 = 1.0 + alpha / a_amp, -2.0 * cw, 1.0 - alpha / a_amp
    return np.array([[b0 / a0, b1 / a0, b2 / a0, 1.0, a1 / a0, a2 / a0]], dtype=np.float64)

_WARMTH_SOS_EVAL = np.vstack([
    _biquad_eval(235.0, +2.8, 0.85),
    _biquad_eval(460.0, +1.8, 1.00),
    _biquad_eval(1680.0, -3.6, 1.05),
    _biquad_eval(2680.0, -3.2, 1.15),
])

def _norm_eval(s):
    return re.sub(r"\s+", " ", re.sub(r"[^a-zà-öø-ÿ0-9\s]", "", s.lower())).strip()

base_tts = Qwen3TTSModel.from_pretrained(base_dir, dtype=torch.bfloat16, attn_implementation="sdpa", device_map="cuda:0")
w_asr = WhisperModel("small", device="cpu", compute_type="int8")
for key, y, sr, txt, ttfa, rtf in outs:
    y24 = torchaudio.functional.resample(torch.from_numpy(y).float(), sr, 24000).numpy() if sr != 24000 else y
    with torch.no_grad():
        emb = F.normalize(base_tts.model.extract_speaker_embedding(y24.astype(np.float32), 24000).float().cpu().flatten(), dim=0)
    sim = float((emb * vn_centroid).sum())
    m_raw = _measure_8d(y24)
    y_eq = sig.sosfilt(_WARMTH_SOS_EVAL, y24.astype(np.float64)).astype(np.float32)
    m_eq = _measure_8d(y_eq)
    sf.write(f"/content/eval_ptbr_final/{key}_warmth_eq.wav", np.clip(y_eq, -0.95, 0.95), 24000, subtype="PCM_16")
    y16 = torchaudio.functional.resample(torch.from_numpy(y24).float(), 24000, 16000).numpy().astype(np.float32)
    segs, info = w_asr.transcribe(y16, beam_size=4)
    asr_t = " ".join(s.text.strip() for s in segs)
    w_sim = difflib.SequenceMatcher(None, _norm_eval(txt), _norm_eval(asr_t)).ratio() if key.startswith("pt_") else 1.0
    print(f"{key:9s} [RAW] | TTFA={ttfa:4.0f}ms RTF={rtf:.2f} | HNR={m_raw['hnr']:5.2f} F0={m_raw['f0']:5.1f} F1/2/3={m_raw['f1']:.0f}/{m_raw['f2']:.0f}/{m_raw['f3']:.0f} L/H={m_raw['low_hi']:+5.2f} H12={m_raw['h1_h2']:+5.2f} | ECAPA={sim:.4f} | Wh({info.language}:{w_sim:.2f}) {asr_t}")
    print(f"{key:9s} [EQ ] |                           | HNR={m_eq['hnr']:5.2f} F0={m_eq['f0']:5.1f} F1/2/3={m_eq['f1']:.0f}/{m_eq['f2']:.0f}/{m_eq['f3']:.0f} L/H={m_eq['low_hi']:+5.2f} H12={m_eq['h1_h2']:+5.2f}")
del base_tts, w_asr
gc.collect(); torch.cuda.empty_cache()

In [ ]:
# ==============================================================================
# PASSO 6 (Opcional): Salvar Modelo Treinado v2 no Google Drive ou Hugging Face Hub
# ==============================================================================
import os, shutil

MERGED_DIR = "/content/kurisu_ptbr_v3_v2_merged" if os.path.exists("/content/kurisu_ptbr_v3_v2_merged") else "/content/kurisu_ptbr_v3_merged"
SAVE_TO_DRIVE = True
DRIVE_ROOT = "/content/drive/MyDrive/OpenAmadeus"
DRIVE_DST = os.path.join(DRIVE_ROOT, os.path.basename(MERGED_DIR))
EXTRAS_DST = os.path.join(DRIVE_ROOT, "kurisu_ptbr_v2_extras")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_ROOT, exist_ok=True)
    if os.path.exists(DRIVE_DST):
        shutil.rmtree(DRIVE_DST)
    shutil.copytree(
        MERGED_DIR, DRIVE_DST,
        ignore=lambda src, names: [n for n in names if os.path.abspath(src) == os.path.abspath(MERGED_DIR) and n in ("training_data", "training_logs", "training_code", "eval")]
    )
    os.makedirs(EXTRAS_DST, exist_ok=True)
    for src in ("/content/kurisu_ptbr_v2_lora_ckpts", "/content/eval_ptbr_final", "/content/train_log_v2.txt", "/content/ptbr_kurisu_data_v2/bucket_a.jsonl"):
        if os.path.exists(src):
            dst = os.path.join(EXTRAS_DST, os.path.basename(src))
            if os.path.isdir(src):
                if os.path.exists(dst): shutil.rmtree(dst)
                shutil.copytree(src, dst)
            else:
                shutil.copy2(src, dst)
    print("Modelo v2 e extras salvos no Google Drive em:", DRIVE_DST)

PUSH_TO_HF = False
HF_REPO_ID = "seu-usuario/Makise_Kurisu_Qwen3_TTS_PTBR"
HF_TOKEN = ""
if PUSH_TO_HF and HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO_ID, exist_ok=True, private=True)
    api.upload_folder(folder_path=MERGED_DIR, repo_id=HF_REPO_ID, ignore_patterns=["training_data/**", "training_logs/**", "training_code/**", "eval/**"])
    print("Modelo enviado para HF:", HF_REPO_ID)

In [ ]:
# ==============================================================================
# PASSO 7: Subir Servidor OpenAmadeus Worker (FasterQwen3TTS + Cloudflare Tunnel)
# ==============================================================================
import os, asyncio, io, re, time, threading, inspect
from concurrent.futures import ThreadPoolExecutor
import torch
torch.set_grad_enabled(False)
_gpu_lock = threading.Lock()
_gpu_executor = ThreadPoolExecutor(max_workers=1, thread_name_prefix="qwen_cuda")
import soundfile as sf
import numpy as np
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response, StreamingResponse
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare

def _patch_static_layer():
    try:
        from transformers.cache_utils import StaticLayer
        if not hasattr(StaticLayer, "_patched_by_openamadeus"):
            _orig_lazy_init = StaticLayer.lazy_initialization
            if len(inspect.signature(_orig_lazy_init).parameters) == 2:
                def _compat_lazy_init(self, key_states, value_states=None):
                    return _orig_lazy_init(self, key_states)
                StaticLayer.lazy_initialization = _compat_lazy_init
                StaticLayer._patched_by_openamadeus = True
    except Exception:
        pass

_patch_static_layer()

from faster_qwen3_tts import FasterQwen3TTS

import scipy.signal as _sig

MODEL_CANDIDATES = [
    "/content/kurisu_ptbr_v3_v2_merged",
    "/content/drive/MyDrive/OpenAmadeus/kurisu_ptbr_v3_v2_merged",
    "/content/kurisu_ptbr_v3_merged",
    "/content/drive/MyDrive/OpenAmadeus/kurisu_ptbr_v3_merged",
]
MODEL_TO_SERVE = next((p for p in MODEL_CANDIDATES if os.path.exists(p)), MODEL_CANDIDATES[0])
WARMTH_EQ = False

def _w_biquad(f0: float, gain_db: float, q: float, fs: int = 24000) -> np.ndarray:
    a_amp = 10.0 ** (gain_db / 40.0)
    w0 = 2.0 * np.pi * f0 / fs
    alpha = np.sin(w0) / (2.0 * q)
    cos_w0 = np.cos(w0)
    b0, b1, b2 = 1.0 + alpha * a_amp, -2.0 * cos_w0, 1.0 - alpha * a_amp
    a0, a1, a2 = 1.0 + alpha / a_amp, -2.0 * cos_w0, 1.0 - alpha / a_amp
    return np.array([[b0 / a0, b1 / a0, b2 / a0, 1.0, a1 / a0, a2 / a0]], dtype=np.float64)

_KURISU_WARMTH_SOS = np.vstack([
    _w_biquad(235.0, +2.8, 0.85),
    _w_biquad(460.0, +1.8, 1.00),
    _w_biquad(1680.0, -3.6, 1.05),
    _w_biquad(2680.0, -3.2, 1.15),
])

def _apply_warmth_chunk(chunk: np.ndarray, eq_zi: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    if not WARMTH_EQ or len(chunk) == 0:
        return chunk, eq_zi
    y, next_zi = _sig.sosfilt(_KURISU_WARMTH_SOS, chunk.astype(np.float64), zi=eq_zi)
    return np.clip(y.astype(np.float32), -0.95, 0.95), next_zi
device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8) else torch.float16

model = FasterQwen3TTS.from_pretrained(MODEL_TO_SERVE, device=device, dtype=dtype, max_seq_len=2048)
if getattr(model, "predictor_graph", None) is not None:
    model.predictor_graph.temperature = 0.6
    model.predictor_graph.top_p = 0.9

def _warmup():
    model.warmup(prefill_len=100)
    with _gpu_lock, torch.inference_mode():
        for _ in model.generate_custom_voice_streaming(text="Oi.", speaker="kurisu", language="Portuguese", instruct=None, chunk_size=8):
            pass

_gpu_executor.submit(_warmup).result()
print("Pipeline PT-BR aquecido:", MODEL_TO_SERVE)

def apply_boundary_continuity(chunk: np.ndarray, prev_sample: Optional[float], ramp_len: int = 120) -> np.ndarray:
    if prev_sample is None or len(chunk) == 0:
        return chunk
    actual_ramp = min(ramp_len, len(chunk))
    start_jump = float(prev_sample) - float(chunk[0])
    ramp = np.linspace(1.0, 0.0, actual_ramp, dtype=chunk.dtype)
    chunk[:actual_ramp] += start_jump * ramp
    return chunk

def sanitize_for_qwen(text: str) -> str:
    s = re.sub(r"<!--[^>]*-->", "", text or "")
    s = re.sub(r"\*[^*]+\*", "", s)
    s = re.sub(r'''[*_~`#"'«»]''', "", s)
    s = re.sub(r"[…‥]", "...", s)
    s = re.sub(r"\?+!+[!?]*|!+\?+[!?]*", "?", s)
    s = re.sub(r"!+", ".", s)
    s = re.sub(r"([.,])\s*\.", r"\1", s)
    s = re.sub(r"\s*\.{2,}\s*$", ".", s)
    s = re.sub(r"\s*\.{2,}\s*", ", ", s)
    s = re.sub(r"\s+", " ", s).strip()
    if s and s[0].islower(): s = s[0].upper() + s[1:]
    if s and s[-1] not in ".?": s += "."
    return s

def prepare_worker_segments(text: str, max_single_pass_chars: int = 175) -> list[str]:
    cleaned = sanitize_for_qwen(text)
    if not cleaned: return []
    if len(cleaned) <= max_single_pass_chars: return [cleaned]
    raw_parts = [p.strip() for p in re.split(r"(?<=[.?!])\s+", cleaned) if p.strip()]
    segments = []
    for part in raw_parts:
        if part[-1] not in ".?": part += "."
        if segments and (len(segments[-1]) + 1 + len(part) <= max_single_pass_chars or len(part) < 25):
            segments[-1] = f"{segments[-1]} {part}"
        else:
            segments.append(part)
    return segments or [cleaned]

app = FastAPI(title="OpenAmadeus Kurisu PT-BR Worker")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class SynthesizeRequest(BaseModel):
    text: str
    language: Optional[str] = "Portuguese"
    speaker: Optional[str] = "kurisu"
    instruct: Optional[str] = None
    temperature: Optional[float] = 0.70
    top_p: Optional[float] = 0.85
    repetition_penalty: Optional[float] = 1.05
    max_new_tokens: Optional[int] = 2048

@app.get("/health")
def health():
    vram = torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
    return {"status": "ready", "device": device, "backend": "faster_qwen3_tts", "vram_allocated_mb": round(vram, 2), "model": MODEL_TO_SERVE}

_active_cancels: list[threading.Event] = []
def _new_cancel() -> threading.Event:
    ev = threading.Event()
    _active_cancels.append(ev)
    if len(_active_cancels) > 16:
        _active_cancels.pop(0)
    return ev

@app.post("/abort")
def abort_active():
    for ev in list(_active_cancels):
        ev.set()
    _active_cancels.clear()
    return {"status": "aborted"}

def _iter_audio(segments, req, cancel):
    eff_temp = 0.70 if req.temperature is None else float(min(max(req.temperature, 0.55), 0.85))
    params = dict(
        speaker=req.speaker or "kurisu",
        language=req.language or "Portuguese",
        instruct=None,
        temperature=eff_temp,
        top_p=min(req.top_p if req.top_p is not None else 0.85, 0.90),
        repetition_penalty=max(req.repetition_penalty if req.repetition_penalty is not None else 1.05, 1.05),
        max_new_tokens=req.max_new_tokens if req.max_new_tokens is not None else 2048,
    )
    prev_sample = None
    for seg_idx, seg_text in enumerate(segments):
        if cancel.is_set(): return
        if seg_idx > 0:
            pause_chunk = apply_boundary_continuity(np.zeros(int(24000 * 0.05), dtype=np.float32), prev_sample)
            prev_sample = float(pause_chunk[-1])
            yield pause_chunk
        stream_gen = model.generate_custom_voice_streaming(text=seg_text, chunk_size=8, **params)
        eq_zi = np.zeros((_KURISU_WARMTH_SOS.shape[0], 2), dtype=np.float64)
        try:
            for audio_chunk, _, _ in stream_gen:
                if cancel.is_set(): break
                raw_chunk = np.asarray(audio_chunk, dtype=np.float32).copy()
                eq_chunk, eq_zi = _apply_warmth_chunk(raw_chunk, eq_zi)
                chunk = apply_boundary_continuity(eq_chunk, prev_sample)
                prev_sample = float(chunk[-1])
                yield chunk
        finally:
            stream_gen.close()

def _to_pcm16(chunk):
    return (np.clip(chunk, -1.0, 1.0) * 32767).astype(np.int16).tobytes()

@app.post("/synthesize_stream")
async def synthesize_stream(req: SynthesizeRequest):
    segments = prepare_worker_segments(req.text or "")
    if not segments: raise HTTPException(status_code=400, detail="Texto vazio.")
    cancel = _new_cancel()
    loop = asyncio.get_running_loop()
    queue = asyncio.Queue()
    def produce():
        try:
            with _gpu_lock, torch.inference_mode():
                for chunk in _iter_audio(segments, req, cancel):
                    loop.call_soon_threadsafe(queue.put_nowait, _to_pcm16(chunk))
        finally:
            try: loop.call_soon_threadsafe(queue.put_nowait, None)
            except RuntimeError: pass
    _gpu_executor.submit(produce)
    async def pcm_body():
        try:
            while (pcm := await queue.get()) is not None: yield pcm
        finally:
            cancel.set()
    return StreamingResponse(pcm_body(), media_type="audio/L16;rate=24000", headers={"X-Audio-Format": "pcm_s16le", "X-Sample-Rate": "24000", "X-Backend": "faster_qwen3_tts", "X-Accel-Buffering": "no"})

@app.post("/synthesize")
async def synthesize(req: SynthesizeRequest):
    segments = prepare_worker_segments(req.text or "")
    if not segments: raise HTTPException(status_code=400, detail="Texto vazio.")
    cancel = _new_cancel()
    def _run():
        with _gpu_lock, torch.inference_mode():
            return list(_iter_audio(segments, req, cancel))
    loop = asyncio.get_running_loop()
    pieces = await loop.run_in_executor(_gpu_executor, _run)
    if cancel.is_set() or not pieces: raise HTTPException(status_code=409, detail="Cancelado.")
    buf = io.BytesIO(); sf.write(buf, np.concatenate(pieces), 24000, format="WAV", subtype="PCM_16"); buf.seek(0)
    return Response(content=buf.read(), media_type="audio/wav", headers={"X-Backend": "faster_qwen3_tts"})

PORT = 8005
tunnel_res = try_cloudflare(port=PORT)
tunnel_url = tunnel_res.tunnel if hasattr(tunnel_res, 'tunnel') else str(tunnel_res)
print("\n" + "=" * 65)
print(f"  URL DO WORKER KURISU PT-BR: {tunnel_url}")
print("  Cole esta URL em Configuracoes > Voz > URL Remota (Colab) no OpenAmadeus!")
print("=" * 65 + "\n")
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)
await server.serve()